# Collections Module

> Generated from [`01-Collections-Module.py`](01-Collections-Module.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
The `collections` module — specialized container types that solve common problems the built-in `dict`/`list` handle awkwardly: counting, grouping, queues, and readable tuples.

### Key points
- **`Counter`** — counts hashable items. `counts["missing"]` returns `0` instead of raising `KeyError`. `.most_common(n)` gives the top N.
- **`Counter` arithmetic** — `+ - & |` act like set/multiset ops (add, subtract, min, max) and drop zero/negative results; `.subtract()` mutates in place and *does* allow negatives.
- **`defaultdict(factory)`** — auto-creates a value (via `list`, `int`, or any zero-arg callable) the first time a key is touched, so no manual "if key not in dict" checks.
- **`deque`** — double-ended queue, O(1) append/pop from *both* ends (a plain `list` is O(n) at the front). `dq.rotate(n)` shifts elements around.
- **`deque(maxlen=N)`** — fixed-size rolling buffer; once full, adding an item silently drops the oldest one. Good for "last N events".
- **`namedtuple`** — tuple with named fields (`p.x` instead of `p[0]`). `._asdict()`, `._replace()` (returns a new one, doesn't mutate), `._fields`.
- **`OrderedDict`** — regular dicts keep insertion order since 3.7 too, but `OrderedDict` adds extras like `.move_to_end()`.
- **`ChainMap`** — layers multiple dicts into one view; lookups check the first dict first, falling back to the next. Good for config overrides + defaults.
- **`UserDict`/`UserList`/`UserString`** — wrap a real dict/list/str internally, meant for subclassing when you want to override methods reliably.

### Gotchas (the part worth revising twice)
- Subclassing `dict`/`list` directly can misbehave — some built-in methods bypass your overrides. Use `UserDict`/`UserList`/`UserString` instead when you need reliable overriding.
- `Counter - Counter` (operator) silently drops zero/negative counts; `Counter.subtract()` does the opposite and keeps negatives. Easy to mix these up.
- `namedtuple` subclasses should set `__slots__ = ()` to stay lightweight (no per-instance `__dict__`).
- `defaultdict`'s factory is any zero-arg callable, not just `list`/`int` — you can pass a custom function that returns a whole dict.

### When it matters
Reach for these whenever a plain dict/list starts needing "if missing, do X" logic, when you're tallying/counting things, or when you need a queue/rolling buffer without hand-rolling one.

### One-line recap example
```python
from collections import Counter, defaultdict
counts = Counter(["a", "b", "a"])       # Counter({'a': 2, 'b': 1})
groups = defaultdict(list)
groups["x"].append(1)                   # no KeyError, auto-creates the list
```

In [ ]:
from collections import Counter, defaultdict, deque, namedtuple, OrderedDict, ChainMap

## 01. collections Module - Introduction

- Provides specialized container datatypes as alternatives to Python's
  general-purpose built-ins (dict, list, set, tuple).

## 02. Counter - Counting Hashable Objects

In [ ]:
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]
counts = Counter(words)
print("\n# 02. Counter")
print(counts)
print("most common:", counts.most_common(2))
print("count of 'apple':", counts["apple"])
print("count of missing key (no KeyError):", counts["mango"])

## 03. defaultdict - Dict with Default Values

In [ ]:
groups = defaultdict(list)
pairs = [("fruit", "apple"), ("veg", "carrot"), ("fruit", "banana")]
for category, item in pairs:
    groups[category].append(item)
print("\n# 03. defaultdict")
print(dict(groups))

word_lengths = defaultdict(int)
for word in words:
    word_lengths[word] += 1
print(dict(word_lengths))

## 04. deque - Double-Ended Queue

- Fast O(1) appends/pops from both ends (list is O(n) at the front).

In [ ]:
dq = deque([1, 2, 3])
dq.append(4)       # add to right
dq.appendleft(0)   # add to left
print("\n# 04. deque")
print(dq)
dq.pop()           # remove from right
dq.popleft()       # remove from left
print(dq)

dq.rotate(1)
print("rotated right:", dq)

## 05. namedtuple - Tuples with Named Fields

In [ ]:
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print("\n# 05. namedtuple")
print(p)
print("x:", p.x, "y:", p.y)
print("as dict:", p._asdict())

## 06. OrderedDict - Dict that Remembers Insertion Order

- Note: regular dicts preserve insertion order since Python 3.7,
  but OrderedDict offers extra methods like move_to_end().

In [ ]:
od = OrderedDict()
od["b"] = 2
od["a"] = 1
od["c"] = 3
print("\n# 06. OrderedDict")
print(od)
od.move_to_end("b")
print("after move_to_end('b'):", od)

## 07. ChainMap - Combine Multiple Dicts into One View

In [ ]:
defaults = {"theme": "light", "language": "en"}
overrides = {"theme": "dark"}
combined = ChainMap(overrides, defaults)
print("\n# 07. ChainMap")
print("theme:", combined["theme"])       # overrides wins
print("language:", combined["language"])  # falls back to defaults

## 08. Counter Arithmetic - +, -, &, |

- Counters support set-like/arithmetic operators. Results drop
  zero/negative counts (except subtract(), see #09).

In [ ]:
stock = Counter(apple=3, banana=1, cherry=0)
sold = Counter(apple=1, banana=2, cherry=5)
print("\n# 08. Counter Arithmetic")
print("stock + sold:", stock + sold)   # add counts
print("stock - sold:", stock - sold)   # subtract, drop <= 0 results
print("stock & sold:", stock & sold)   # min of each count (intersection)
print("stock | sold:", stock | sold)   # max of each count (union)

## 09. Counter.subtract() - In-Place Subtraction (allows negatives)

In [ ]:
running_total = Counter(apple=3, banana=1)
running_total.subtract(Counter(apple=1, banana=2))
print("\n# 09. Counter.subtract()")
print(running_total)  # banana goes negative, unlike the '-' operator

## 10. deque(maxlen=N) - Fixed-Size Rolling Buffer

- Once full, appending drops the oldest item automatically.
- Useful for "last N events" style tracking (recent logs, undo history).

In [ ]:
recent_events = deque(maxlen=3)
print("\n# 10. deque(maxlen=N)")
for event in ["login", "click", "scroll", "logout"]:
    recent_events.append(event)
    print(f"after '{event}':", list(recent_events))

## 11. defaultdict with a Custom Factory Function

- The factory isn't limited to list/int; any zero-arg callable works.

In [ ]:
def new_player():
    return {"score": 0, "lives": 3}

players = defaultdict(new_player)
print("\n# 11. defaultdict with Custom Factory")
players["Alice"]["score"] += 10
players["Bob"]["lives"] -= 1
print(dict(players))

## 12. namedtuple Deeper - _replace(), _fields, Subclassing

In [ ]:
p_moved = p._replace(x=10)          # returns a NEW namedtuple, p is unchanged
print("original p:", p, "| p_moved:", p_moved)
print("field names:", Point._fields)

class Vector(namedtuple("Vector", ["x", "y"])):
    __slots__ = ()  # keep it lightweight, no per-instance __dict__

    def magnitude(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

v = Vector(3, 4)
print("Vector:", v, "| magnitude:", v.magnitude())

## 13. UserDict / UserList / UserString - Subclassing Container-Like Types

- Subclassing dict/list/str directly can behave oddly (some built-in
  methods bypass your overrides). UserDict/UserList/UserString wrap a
  real dict/list/str internally, so overriding methods works reliably.

In [ ]:
from collections import UserDict

class TrackedDict(UserDict):
    def __setitem__(self, key, value):
        print(f"  setting {key} = {value}")
        super().__setitem__(key, value)

print("\n# 13. UserDict / UserList / UserString")
td = TrackedDict()
td["a"] = 1
td["b"] = 2
print("final dict:", dict(td))